# LeetCode 865 — Smallest Subtree with all the Deepest Nodes bằng Python

Link đề: https://leetcode.com/problems/smallest-subtree-with-all-the-deepest-nodes/ · Độ khó: **Medium** · Chủ đề: Tree, DFS, BFS, Hash Table, Binary Tree, Lowest Common Ancestor

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right

class Solution:
    def subtreeWithAllDeepest(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        ...
```

Ghi chú: hàm trả về **một nút** (và cả cây con của nó). Test so sánh **theo danh tính** (`is`), không so theo giá trị. Đề này giống bài 1123 (*Lowest Common Ancestor of Deepest Leaves*). Notebook tự dựng `TreeNode` và hàm dựng cây từ danh sách theo tầng kiểu LeetCode. Oracle: **tính theo đúng định nghĩa** — liệt kê mọi nút, tập nút sâu nhất, tập con của từng nút, rồi chọn cây con **nhỏ nhất** chứa mọi nút sâu nhất.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: độ sâu của nút là khoảng cách tới gốc; tìm cây con nhỏ nhất chứa **mọi** nút sâu nhất — thực chất là tổ tiên chung thấp nhất (LCA) của các nút sâu nhất.
2. Nắm công cụ Python: lớp `TreeNode`, dựng cây theo tầng, đệ quy trả về **cặp** `(độ sâu, nút)`, `dict` nhớ chiều cao theo nút, BFS theo tầng, bản đồ cha.
3. Đi từ **định nghĩa trực tiếp (kiểm từng nút có chứa hết nút sâu nhất)** → **một lượt DFS trả `(độ sâu, nút)`** → **tính chiều cao rồi đi xuống nhánh sâu hơn** → **BFS tầng cuối rồi leo lên theo cha**.
4. Hiểu mấu chốt: nếu hai cây con có **cùng** độ sâu tối đa thì nút hiện tại là đáp án; nếu một bên sâu hơn thì đáp án nằm hoàn toàn trong bên đó.
5. Tránh lỗi kinh điển: khi hoà vẫn đi sang trái, trả về lá sâu nhất đầu tiên thay vì LCA, tính LCA của mọi lá (không chỉ lá sâu nhất), nhầm "chiều cao" với "độ sâu", và so sánh nút theo giá trị.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân. **Độ sâu** của một nút là khoảng cách ngắn nhất từ nó tới gốc (gốc có độ sâu `0`).

Trả về **cây con nhỏ nhất** chứa **tất cả các nút sâu nhất** (nút có độ sâu lớn nhất trong cả cây).

**Cây con** của một nút gồm nút đó và mọi hậu duệ.

**Ví dụ 1:** `root = [3,5,1,6,2,0,8,null,null,7,4]` → `[2,7,4]` (nút `2`; các nút sâu nhất là `7` và `4`).

**Ví dụ 2:** `root = [1]` → `[1]`.

**Ví dụ 3:** `root = [0,1,3,null,2]` → `[2]` (nút sâu nhất duy nhất là `2`).

**Ràng buộc:** số nút `[1, 500]`; `0 <= Node.val <= 500`; giá trị **duy nhất**.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None là nút trống)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def to_level(root):
    """Danh sách theo tầng (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            q.append(node.left)
            q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def all_nodes(root):
    """Mọi nút (tiền thứ tự, lặp, không đệ quy)."""
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node)
        if node.right:
            st.append(node.right)
        if node.left:
            st.append(node.left)
    return out


examples = [
    ([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], [2, 7, 4]),
    ([1], [1]),
    ([0, 1, 3, None, 2], [2]),
]
for vals, want in examples:
    r = build_tree(vals)
    assert to_level(r) == vals
    print(vals, "-> cây con mong đợi", want)

[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4] -> cây con mong đợi [2, 7, 4]
[1] -> cây con mong đợi [1]
[0, 1, 3, None, 2] -> cây con mong đợi [2]


## 3. Kiến thức nền

### 3.1 Độ sâu và chiều cao

- **Độ sâu** của nút: số cạnh từ gốc xuống nút (gốc: 0).
- **Chiều cao** của nút: số tầng của cây con của nó (hoặc số cạnh tới lá xa nhất). Ta dùng quy ước: `height(None) = 0`, `height(node) = 1 + max(height(left), height(right))`.

Nút sâu nhất là những nút có độ sâu bằng `chiều cao cây − 1`.

In [2]:
def depths(root):
    d = {root: 0}
    st = [root]
    while st:
        node = st.pop()
        for ch in (node.left, node.right):
            if ch:
                d[ch] = d[node] + 1
                st.append(ch)
    return d


r = build_tree(examples[0][0])
dep = depths(r)
mx = max(dep.values())
print("độ sâu lớn nhất:", mx, "| nút sâu nhất:", sorted(n.val for n, v in dep.items() if v == mx))

độ sâu lớn nhất: 3 | nút sâu nhất: [4, 7]


### 3.2 Cây con và danh tính nút

Nút `node` đại diện cho cây con gốc tại `node`. Hai nút có thể có cùng giá trị trong cây tuỳ ý, nên ta so sánh nút bằng `is` (danh tính). Đề đảm bảo giá trị duy nhất, nhưng làm theo danh tính là an toàn hơn.

### 3.3 LCA — tổ tiên chung thấp nhất

LCA của một tập nút là nút **sâu nhất** là tổ tiên (cho phép chính nó) của **mọi** nút trong tập. Cây con nhỏ nhất chứa cả tập chính là cây con gốc tại LCA. Vậy bài này là: **LCA của tập các nút sâu nhất**.

### 3.4 Đệ quy trả về cặp

Một hàm đệ quy có thể trả về **hai** thông tin cùng lúc (ở đây chiều cao và nút ứng viên) bằng tuple: `h, node = dfs(child)`.

In [3]:
def pair_demo(n):
    return n * 2, f"nút {n}"


h, label = pair_demo(3)
print(h, label)

6 nút 3


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1:

```
        3
      /   \
     5     1
    / \   / \
   6   2 0   8
      / \
     7   4
```

Nút sâu nhất: `7` và `4` (độ sâu 3). Chúng đều nằm trong cây con của `2`, `5`, `3`. Cây con nhỏ nhất chứa cả hai là cây con của `2`.

Nhìn từ dưới lên: tại mỗi nút, hỏi cây con trái sâu bao nhiêu, cây con phải sâu bao nhiêu:

- Nếu **trái sâu hơn**: mọi nút sâu nhất nằm ở trái → đáp án ở trái.
- Nếu **phải sâu hơn**: đáp án ở phải.
- Nếu **bằng nhau**: nút sâu nhất có ở cả hai phía → nút hiện tại chính là đáp án.

Tại `2`: trái `7` (sâu 1), phải `4` (sâu 1): bằng nhau → `2`. Tại `5`: trái `6` có chiều cao 1, phải `2` có chiều cao 2 → phải sâu hơn → đáp án là đáp án của `2`. Tại `3`: trái (cây `5`) chiều cao 3, phải (cây `1`) chiều cao 2 → trái → đáp án của `5` = `2` ✓.

In [4]:
def height(node):
    return 0 if node is None else 1 + max(height(node.left), height(node.right))


r = build_tree(examples[0][0])
for v in (3, 5, 1, 2):
    node = next(n for n in all_nodes(r) if n.val == v)
    print(f"nút {v}: chiều cao trái = {height(node.left)}, phải = {height(node.right)}")

nút 3: chiều cao trái = 3, phải = 2
nút 5: chiều cao trái = 1, phải = 2
nút 1: chiều cao trái = 1, phải = 1
nút 2: chiều cao trái = 1, phải = 1


## 5. Cách 1 — Brute Force: theo định nghĩa

**Ý tưởng:** tính độ sâu mọi nút, lấy tập `D` các nút sâu nhất. Với **từng** nút `x` ta liệt kê tập nút của cây con `x` và kiểm tra `D ⊆ subtree(x)`. Các nút thoả tạo thành một **chuỗi** từ gốc xuống; đáp án là nút **sâu nhất** (độ sâu lớn nhất) trong chuỗi.

In [5]:
def subtree_nodes(node):
    return set(all_nodes(node))


def deepest_brute(root):
    dep = depths(root)
    mx = max(dep.values())
    deepest = {n for n, d in dep.items() if d == mx}
    best, best_depth = None, -1
    for x in all_nodes(root):
        if deepest <= subtree_nodes(x):                  # cây con của x chứa mọi nút sâu nhất
            if dep[x] > best_depth:
                best, best_depth = x, dep[x]
    return best


for vals, want in examples:
    assert to_level(deepest_brute(build_tree(vals))) == want
print([to_level(deepest_brute(build_tree(v))) for v, _ in examples])

[[2, 7, 4], [1], [2]]


**Độ phức tạp:** với mỗi nút `x`, liệt kê cây con `O(n)` và kiểm tập con `O(|D|)` → `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n ≤ 500` chạy được, nhưng lãng phí.

**Nút thắt:** ta kiểm tra từng nút độc lập. Thông tin "cây con này sâu bao nhiêu" có thể tính **một lần từ dưới lên** và dùng ngay.

## 6. Ý tưởng thuật toán / bước đột phá

Tính từ dưới lên với mỗi nút trả về `(chiều cao, đáp án của cây con)`:

- Lá/`None`: `(0, None)` hoặc lá: `(1, lá)`.
- Gọi `(hl, nl)`, `(hr, nr)` của hai con.
- `hl > hr` → `(hl + 1, nl)`; `hl < hr` → `(hr + 1, nr)`; **`hl == hr` → `(hl + 1, node)`**.

Đáp án là nút trong kết quả của gốc. Một lượt `O(n)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** một lượt DFS trả `(chiều cao, nút)`, **Cách 3** tính chiều cao rồi đi xuống nhánh sâu hơn, **Cách 4** BFS tầng cuối rồi leo lên theo cha.

### Cách 2 — Một lượt DFS trả `(chiều cao, nút)`

**Giải thích từng đoạn:**

1. `dfs(None)` trả `(0, None)`.
2. Với `node`: `hl, nl = dfs(left)`; `hr, nr = dfs(right)`.
3. Chiều cao lớn hơn thắng; nếu bằng nhau thì `node` là ứng viên.

In [6]:
def deepest_pair(root):
    def dfs(node):
        if node is None:
            return 0, None
        hl, nl = dfs(node.left)
        hr, nr = dfs(node.right)
        if hl > hr:
            return hl + 1, nl
        if hl < hr:
            return hr + 1, nr
        return hl + 1, node                              # hai bên sâu bằng nhau: node là LCA của các nút sâu nhất

    return dfs(root)[1]


for vals, want in examples:
    assert to_level(deepest_pair(build_tree(vals))) == want
print([to_level(deepest_pair(build_tree(v))) for v, _ in examples])

[[2, 7, 4], [1], [2]]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionPair:
    def subtreeWithAllDeepest(self, root):
        def dfs(node):
            if node is None:
                return 0, None
            hl, nl = dfs(node.left)
            hr, nr = dfs(node.right)
            if hl > hr:
                return hl + 1, nl
            if hl < hr:
                return hr + 1, nr
            return hl + 1, node

        return dfs(root)[1]

**Độ phức tạp:** `O(n)` thời gian (mỗi nút một lần), `O(h)` bộ nhớ ngăn xếp đệ quy.

### Cách 3 — Tính chiều cao, rồi đi xuống nhánh sâu hơn

**Ý tưởng:** trước hết tính `height[node]` cho mọi nút (một lượt, lưu vào `dict`). Sau đó từ gốc: nếu `height(left) > height(right)` đi trái, `<` đi phải, bằng thì dừng — nút đó là đáp án.

In [8]:
def deepest_walk(root):
    height_of = {None: 0}

    def calc(node):
        if node is None:
            return 0
        h = 1 + max(calc(node.left), calc(node.right))
        height_of[node] = h
        return h

    calc(root)
    node = root
    while True:
        hl, hr = height_of[node.left], height_of[node.right]
        if hl > hr:
            node = node.left
        elif hl < hr:
            node = node.right
        else:
            return node                                  # hai bên bằng nhau (kể cả cả hai rỗng): đây là đáp án


for vals, want in examples:
    assert to_level(deepest_walk(build_tree(vals))) == want
print([to_level(deepest_walk(build_tree(v))) for v, _ in examples])

[[2, 7, 4], [1], [2]]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionWalk:
    def subtreeWithAllDeepest(self, root):
        height = {None: 0}

        def calc(node):
            if node is None:
                return 0
            h = 1 + max(calc(node.left), calc(node.right))
            height[node] = h
            return h

        calc(root)
        node = root
        while True:
            hl, hr = height[node.left], height[node.right]
            if hl > hr:
                node = node.left
            elif hl < hr:
                node = node.right
            else:
                return node

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ cho `dict` chiều cao.

### Cách 4 — BFS tầng cuối rồi leo lên theo cha

**Ý tưởng:** dựng bản đồ cha trong khi BFS theo tầng; lấy **tầng cuối** (mọi nút sâu nhất). Rồi lặp: thay tập hiện tại bằng tập **cha** của chúng (bỏ trùng); khi chỉ còn một nút thì đó là LCA.

In [10]:
def deepest_bfs_climb(root):
    parent = {root: None}
    level = [root]
    while True:
        nxt = []
        for node in level:
            for ch in (node.left, node.right):
                if ch:
                    parent[ch] = node
                    nxt.append(ch)
        if not nxt:
            break
        level = nxt                                       # tầng sâu nhất tính đến lúc này
    current = set(level)
    while len(current) > 1:
        current = {parent[n] for n in current}            # leo lên một tầng, gộp các cha trùng nhau
    return next(iter(current))


for vals, want in examples:
    assert to_level(deepest_bfs_climb(build_tree(vals))) == want
print([to_level(deepest_bfs_climb(build_tree(v))) for v, _ in examples])

[[2, 7, 4], [1], [2]]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionClimb:
    def subtreeWithAllDeepest(self, root):
        parent = {root: None}
        level = [root]
        while True:
            nxt = []
            for node in level:
                for ch in (node.left, node.right):
                    if ch:
                        parent[ch] = node
                        nxt.append(ch)
            if not nxt:
                break
            level = nxt
        current = set(level)
        while len(current) > 1:
            current = {parent[n] for n in current}
        return next(iter(current))

**Độ phức tạp:** `O(n)` thời gian (BFS `O(n)`; leo lên: tổng số phần tử các tập `≤ n`), `O(n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với ví dụ 1. Giá trị `dfs(node)` (chiều cao, nút) theo thứ tự hậu tự:

| nút | `(hl, nl)` | `(hr, nr)` | kết quả `(h, nút)` |
|---|---|---|---|
| `6` | `(0, None)` | `(0, None)` | `(1, 6)` (hoà → chính nó) |
| `7` | `(0, None)` | `(0, None)` | `(1, 7)` |
| `4` | `(0, None)` | `(0, None)` | `(1, 4)` |
| `2` | `(1, 7)` | `(1, 4)` | `(2, 2)` (hoà) |
| `5` | `(1, 6)` | `(2, 2)` | `(3, 2)` (phải sâu hơn) |
| `0`, `8` | `(0,None)` | `(0,None)` | `(1, 0)`, `(1, 8)` |
| `1` | `(1, 0)` | `(1, 8)` | `(2, 1)` (hoà) |
| `3` | `(3, 2)` | `(2, 1)` | `(4, 2)` (trái sâu hơn) |

Kết quả: nút `2` ✓. Hàm trace in đúng các dòng này:

In [12]:
def trace_pair(root):
    def dfs(node):
        if node is None:
            return 0, None
        hl, nl = dfs(node.left)
        hr, nr = dfs(node.right)
        if hl > hr:
            res, why = (hl + 1, nl), "trái sâu hơn"
        elif hl < hr:
            res, why = (hr + 1, nr), "phải sâu hơn"
        else:
            res, why = (hl + 1, node), "hoà -> chính nó"
        show = lambda p: (p[0], p[1].val if p[1] else None)
        print(f"nút {node.val:>2}: trái {show((hl, nl))}, phải {show((hr, nr))} -> {show(res)}  ({why})")
        return res

    ans = dfs(root)[1]
    print("=> đáp án: nút", ans.val)
    return ans


assert trace_pair(build_tree(examples[0][0])).val == 2

nút  6: trái (0, None), phải (0, None) -> (1, 6)  (hoà -> chính nó)
nút  7: trái (0, None), phải (0, None) -> (1, 7)  (hoà -> chính nó)
nút  4: trái (0, None), phải (0, None) -> (1, 4)  (hoà -> chính nó)
nút  2: trái (1, 7), phải (1, 4) -> (2, 2)  (hoà -> chính nó)
nút  5: trái (1, 6), phải (2, 2) -> (3, 2)  (phải sâu hơn)
nút  0: trái (0, None), phải (0, None) -> (1, 0)  (hoà -> chính nó)
nút  8: trái (0, None), phải (0, None) -> (1, 8)  (hoà -> chính nó)
nút  1: trái (1, 0), phải (1, 8) -> (2, 1)  (hoà -> chính nó)
nút  3: trái (3, 2), phải (2, 1) -> (4, 2)  (trái sâu hơn)
=> đáp án: nút 2


**Trace Cách 4 (BFS + leo cha)** với ví dụ 1: tầng cuối là `{7, 4}`. Leo một bước: cha `{2}` → chỉ còn một nút → đáp án `2` ✓. Với cây `[0,1,3,null,2]`: tầng cuối `{2}` → đã một nút → đáp án `2` (không cần leo).

In [13]:
def trace_climb(root):
    parent = {root: None}
    level = [root]
    while True:
        nxt = []
        for node in level:
            for ch in (node.left, node.right):
                if ch:
                    parent[ch] = node
                    nxt.append(ch)
        if not nxt:
            break
        level = nxt
    current = set(level)
    print("tầng cuối:", sorted(n.val for n in current))
    while len(current) > 1:
        current = {parent[n] for n in current}
        print("leo lên  :", sorted(n.val for n in current))
    ans = next(iter(current))
    print("=> đáp án: nút", ans.val)
    return ans


assert trace_climb(build_tree(examples[0][0])).val == 2
print()
assert trace_climb(build_tree(examples[2][0])).val == 2

tầng cuối: [4, 7]
leo lên  : [2]
=> đáp án: nút 2

tầng cuối: [2]
=> đáp án: nút 2


## 9. Code nộp LeetCode

Chọn **Cách 2** (một lượt, `O(n)`, ngắn gọn). Giữ nguyên `class Solution`, `subtreeWithAllDeepest(self, root)`:

In [14]:
class Solution:
    def subtreeWithAllDeepest(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        def dfs(node):
            if node is None:
                return 0, None                            # (chiều cao, đáp án của cây con này)
            hl, nl = dfs(node.left)
            hr, nr = dfs(node.right)
            if hl > hr:
                return hl + 1, nl                         # nút sâu nhất đều nằm bên trái
            if hl < hr:
                return hr + 1, nr                         # nút sâu nhất đều nằm bên phải
            return hl + 1, node                           # hai bên sâu bằng nhau: node là LCA

        return dfs(root)[1]


for vals, want in examples:
    assert to_level(Solution().subtreeWithAllDeepest(build_tree(vals))) == want
print([to_level(Solution().subtreeWithAllDeepest(build_tree(v))) for v, _ in examples])    # [[2,7,4],[1],[2]]

[[2, 7, 4], [1], [2]]


## 10. Vì sao đúng

**(a) Cây con nhỏ nhất = cây con của LCA.** Cây con của `x` chứa tập `D` khi và chỉ khi `x` là tổ tiên (hoặc bằng) của mọi nút trong `D`. Các `x` như vậy tạo thành một **chuỗi** từ gốc tới LCA của `D` (mọi tổ tiên của LCA cũng là tổ tiên chung). Cây con nhỏ nhất là của phần tử cuối chuỗi: chính LCA.

**(b) Cách 2: bất biến.** `dfs(node)` trả `(h, ans)` với `h` là chiều cao cây con và `ans` là LCA của các nút **sâu nhất của cây con `node`** (tính theo độ sâu tương đối trong cây con đó). Chứng minh quy nạp:
- Nếu `hl > hr`: các nút sâu nhất của cây con `node` đúng là các nút sâu nhất của cây con trái (chúng ở độ sâu tương đối `hl`, sâu hơn mọi nút bên phải) nên `ans = nl`.
- Nếu `hl < hr`: đối xứng.
- Nếu `hl == hr > 0`: các nút sâu nhất nằm ở **cả hai** phía, nên LCA của chúng là `node`.
- Nếu `hl == hr == 0` (lá): nút sâu nhất của cây con là chính `node`, LCA là `node`.
Ở gốc, độ sâu tương đối chính là độ sâu thật, nên `ans` là đáp án.

**(c) Cách 3.** `height` tính đúng (đệ quy). Đi xuống theo quy tắc ở (b): chừng nào một bên **sâu hơn thì mọi nút sâu nhất nằm trọn trong bên đó**; dừng khi hoà.

**(d) Cách 4.** Tầng cuối BFS đúng là tập các nút sâu nhất. Tập các cha của một tập nút `S` (cùng độ sâu) là tập tổ tiên ở một tầng trên; lặp lại tới khi chỉ còn một nút, nút đó là tổ tiên chung đầu tiên (sâu nhất) của `S`: vì ở mỗi bước các nút trong tập có cùng độ sâu, và tập chỉ co lại thành một khi mọi đường lên gặp nhau.

**(e) Cách 1** là định nghĩa trực tiếp.

Ta kiểm chứng bằng máy: mọi cách khớp **oracle theo định nghĩa (cây con nhỏ nhất chứa tập sâu nhất)** trên mọi hình dạng cây tới 8 nút (mục 12D) và khớp nhau trên cây tới 500 nút (12E–F), so sánh **theo danh tính nút**.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Theo định nghĩa | `O(n²)` | `O(n)` | oracle phụ |
| 2. DFS trả `(h, nút)` | `O(n)` | `O(h)` | **đáp án chính** |
| 3. Chiều cao + đi xuống | `O(n)` | `O(n)` | dễ hiểu từng bước |
| 4. BFS + leo cha | `O(n)` | `O(n)` | không cần đệ quy |

Đo thực tế trên cây 500 nút:

In [15]:
def random_tree(rng, n, chain_bias=0.0):
    """Cây n nút, giá trị duy nhất 0..500; gắn nút mới vào chỗ trống ngẫu nhiên (chain_bias cao -> cây sâu)."""
    vals = rng.sample(range(501), n)
    root = TreeNode(vals[0])
    slots = [(root, "left"), (root, "right")]
    for v in vals[1:]:
        idx = len(slots) - 1 - rng.randrange(min(2, len(slots))) if rng.random() < chain_bias else rng.randrange(len(slots))
        parent, side = slots.pop(idx)
        node = TreeNode(v)
        setattr(parent, side, node)
        slots.append((node, "left"))
        slots.append((node, "right"))
    return root


def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(865)
for label, bias in [("ngẫu nhiên", 0.0), ("sâu (bias 0.95)", 0.95)]:
    root = random_tree(rng, 500, bias)
    row = [f"{name} {bench(fn, root) * 1e3:7.3f} ms" for name, fn in
           [("brute", deepest_brute), ("pair", deepest_pair), ("walk", deepest_walk), ("climb", deepest_bfs_climb)]]
    print(f"{label:16s}: " + " | ".join(row))

ngẫu nhiên      : brute   1.365 ms | pair   0.090 ms | walk   0.111 ms | climb   0.109 ms
sâu (bias 0.95) : brute   4.955 ms | pair   0.089 ms | walk   0.128 ms | climb   0.127 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi hình dạng cây tới 8 nút với oracle theo định nghĩa, (E) ngẫu nhiên có seed, (F) giới hạn 500 nút.

### A. Ví dụ của đề

In [16]:
def oracle(root):
    """Đúng định nghĩa đề: tập nút sâu nhất D; trong mọi nút x có cây con chứa D, chọn cây con NHỎ NHẤT (ít nút nhất)."""
    dep = {}
    st = [(root, 0)]
    while st:
        node, d = st.pop()
        dep[id(node)] = d
        for ch in (node.left, node.right):
            if ch:
                st.append((ch, d + 1))
    mx = max(dep.values())
    D = {i for i, d in dep.items() if d == mx}
    best, best_size = None, None
    for x in all_nodes(root):
        sub = {id(n) for n in all_nodes(x)}
        if D <= sub and (best is None or len(sub) < best_size):
            best, best_size = x, len(sub)
    return best


IMPLS = {
    "brute": deepest_brute,
    "pair": deepest_pair,
    "walk": deepest_walk,
    "climb": deepest_bfs_climb,
    "Solution": lambda r: Solution().subtreeWithAllDeepest(r),
    "SolutionPair": lambda r: SolutionPair().subtreeWithAllDeepest(r),
    "SolutionWalk": lambda r: SolutionWalk().subtreeWithAllDeepest(r),
    "SolutionClimb": lambda r: SolutionClimb().subtreeWithAllDeepest(r),
}


def check(root, names=None):
    want = oracle(root)
    for name in (names or IMPLS):
        got = IMPLS[name](root)
        assert got is want, (name, to_level(root), got.val if got else None, want.val)     # so sánh theo DANH TÍNH


for vals, want in examples:
    r = build_tree(vals)
    assert to_level(oracle(r)) == want
    check(r)
print("ví dụ đề: oracle và", len(IMPLS), "cài đặt cùng trả đúng nút (so sánh bằng `is`)")

ví dụ đề: oracle và 8 cài đặt cùng trả đúng nút (so sánh bằng `is`)


### B. Edge case

- Cây một nút → chính gốc.
- Cây lệch trái / lệch phải (chuỗi): đáp án là lá cuối.
- Cây đầy đủ (mọi lá cùng độ sâu): đáp án là gốc.
- Cây có hai lá sâu nhất ở hai phía của gốc: gốc.
- Lá sâu nhất duy nhất ở sâu bên trái, bên phải nông hơn: đáp án là chính lá đó.
- Hai lá sâu nhất là anh em: đáp án là cha chung.

In [17]:
def pick(root, val):
    return next(n for n in all_nodes(root) if n.val == val)


r = build_tree([1])
assert Solution().subtreeWithAllDeepest(r) is r
chain_l = build_tree([5, 4, None, 3, None, 2, None, 1])
assert Solution().subtreeWithAllDeepest(chain_l) is pick(chain_l, 1)
chain_r = build_tree([1, None, 2, None, 3, None, 4])
assert Solution().subtreeWithAllDeepest(chain_r) is pick(chain_r, 4)
full = build_tree([1, 2, 3, 4, 5, 6, 7])
assert Solution().subtreeWithAllDeepest(full) is full
two_sides = build_tree([1, 2, 3, 4, None, None, 5])
assert Solution().subtreeWithAllDeepest(two_sides) is two_sides
unique_left = build_tree([1, 2, 3, 4, None, None, None, 5])
assert Solution().subtreeWithAllDeepest(unique_left) is pick(unique_left, 5)
siblings = build_tree([1, 2, 3, None, 4, None, None, 5, 6])
assert Solution().subtreeWithAllDeepest(siblings) is pick(siblings, 4)
for t in (r, chain_l, chain_r, full, two_sides, unique_left, siblings):
    check(t)
print("edge case: 7 cây, mọi cài đặt trả đúng nút")

edge case: 7 cây, mọi cài đặt trả đúng nút


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_tie_left(root):                                # hoà thì vẫn đi sang trái (>=)
    def dfs(node):
        if node is None:
            return 0, None
        hl, nl = dfs(node.left)
        hr, nr = dfs(node.right)
        if hl >= hr:
            return hl + 1, nl if nl else node          # SAI: bỏ qua trường hợp hoà
        return hr + 1, nr

    return dfs(root)[1]


def bug_first_deepest_leaf(root):                      # trả lá sâu nhất đầu tiên gặp
    best, best_d = None, -1
    st = [(root, 0)]
    while st:
        node, d = st.pop()
        if d > best_d:
            best, best_d = node, d
        if node.right:
            st.append((node.right, d + 1))
        if node.left:
            st.append((node.left, d + 1))
    return best


def bug_lca_all_leaves(root):                          # LCA của MỌI lá, không chỉ lá sâu nhất
    def dfs(node):
        if node.left is None and node.right is None:
            return node
        l = dfs(node.left) if node.left else None
        r = dfs(node.right) if node.right else None
        if l and r:
            return node
        return l or r

    return dfs(root)


def bug_follow_taller_to_leaf(root):                   # đi theo nhánh sâu hơn tới tận lá, bỏ qua hoà
    def h(n):
        return 0 if n is None else 1 + max(h(n.left), h(n.right))

    node = root
    while node.left or node.right:
        node = node.left if h(node.left) >= h(node.right) else node.right
    return node


def bug_root_if_two_children(root):                    # có hai con thì trả luôn nút đó (không xét độ sâu)
    node = root
    while node.left is None or node.right is None:
        nxt = node.left or node.right
        if nxt is None:
            return node
        node = nxt
    return node


BUGS = [bug_tie_left, bug_first_deepest_leaf, bug_lca_all_leaves, bug_follow_taller_to_leaf, bug_root_if_two_children]
rng = random.Random(7)
cases = [random_tree(rng, rng.randint(1, 10), rng.choice([0.0, 0.6])) for _ in range(400)]
for bug in BUGS:
    bad = next((r for r in cases if bug(r) is not oracle(r)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:26s} cây {to_level(bad)}: ra {bug(bad).val}, đúng {oracle(bad).val}")

bug_tie_left               cây [292, 157, 286, None, 417, None, 92, None, 349, 327, None, 52, 297]: ra 52, đúng 349
bug_first_deepest_leaf     cây [292, 157, 286, None, 417, None, 92, None, 349, 327, None, 52, 297]: ra 52, đúng 349
bug_lca_all_leaves         cây [202, 333, 24, None, 420, None, 37, None, None, None, 274]: ra 202, đúng 274
bug_follow_taller_to_leaf  cây [292, 157, 286, None, 417, None, 92, None, 349, 327, None, 52, 297]: ra 52, đúng 349
bug_root_if_two_children   cây [202, 333, 24, None, 420, None, 37, None, None, None, 274]: ra 202, đúng 274


### D. Vét cạn mọi hình dạng cây nhỏ

Mọi hình dạng cây nhị phân với `n = 1..8` nút (số Catalan `1, 2, 5, 14, 42, 132, 429, 1430`, tổng `2 055` hình dạng), giá trị khác nhau ngẫu nhiên. Mọi cài đặt phải trả **đúng nút** của oracle.

In [19]:
def shapes(n, memo={}):
    if n in memo:
        return memo[n]
    if n == 0:
        out = [None]
    else:
        out = [(l, r) for k in range(n) for l in shapes(k) for r in shapes(n - 1 - k)]
    memo[n] = out
    return out


def from_shape(shape, vals):
    if shape is None:
        return None
    node = TreeNode(next(vals))
    node.left = from_shape(shape[0], vals)
    node.right = from_shape(shape[1], vals)
    return node


assert [len(shapes(n)) for n in range(1, 9)] == [1, 2, 5, 14, 42, 132, 429, 1430]
t0 = time.perf_counter()
rng = random.Random(2024)
cnt = 0
for n in range(1, 9):
    for sh in shapes(n):
        root = from_shape(sh, iter(rng.sample(range(501), n)))
        check(root)
        cnt += 1
print(cnt, "hình dạng cây (n <= 8): 8 cài đặt trả đúng nút oracle;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 2055

2055 hình dạng cây (n <= 8): 8 cài đặt trả đúng nút oracle; 0.1 s


### E. Ngẫu nhiên có seed và tính chất

Cây ngẫu nhiên tới 60 nút (đủ kiểu độ sâu). Thêm **tính chất**: (1) đáp án là tổ tiên (hoặc chính nó) của **mọi** nút sâu nhất; (2) không con nào của đáp án cũng thoả (tính nhỏ nhất): nếu đáp án có hai con thì mỗi con đơn lẻ không chứa hết mọi nút sâu nhất; (3) độ sâu cây con của đáp án (chiều cao) bằng `chiều cao cây − độ sâu của đáp án`.

In [20]:
def is_ancestor(a, b):
    return b in set(all_nodes(a))


rng = random.Random(20240865)
for _ in range(1500):
    root = random_tree(rng, rng.randint(1, 60), rng.choice([0.0, 0.4, 0.9]))
    check(root)
    ans = Solution().subtreeWithAllDeepest(root)
    dep = depths(root)
    mx = max(dep.values())
    deepest = [n for n, d in dep.items() if d == mx]
    sub = set(all_nodes(ans))
    assert all(n in sub for n in deepest)                                     # (1) chứa mọi nút sâu nhất
    for ch in (ans.left, ans.right):                                           # (2) không con nào đủ
        if ch:
            assert not all(n in set(all_nodes(ch)) for n in deepest)
    assert height(ans) == height(root) - dep[ans]                              # (3) độ cao khớp
print("1500 cây ngẫu nhiên: 8 cài đặt trả đúng nút oracle; thoả 3 tính chất (chứa hết, tối thiểu, chiều cao)")

1500 cây ngẫu nhiên: 8 cài đặt trả đúng nút oracle; thoả 3 tính chất (chứa hết, tối thiểu, chiều cao)


### F. Giới hạn 500 nút

Cây 500 nút: ngẫu nhiên, chuỗi trái, chuỗi phải, cây đầy đủ lệch (gần hoàn chỉnh), hai nhánh sâu bằng nhau, một nhánh sâu hơn: mọi cài đặt (trừ Cách 1 quá chậm: chỉ chạy `n ≤ 200`) phải trả cùng một nút.

In [21]:
def chain_tree(n, side, base=0):
    root = TreeNode(base)
    cur = root
    for v in range(base + 1, base + n):
        nxt = TreeNode(v)
        setattr(cur, side, nxt)
        cur = nxt
    return root


def v_tree(depth_left, depth_right):
    root = TreeNode(0)
    root.left = chain_tree(depth_left, "left", 1) if depth_left else None
    root.right = chain_tree(depth_right, "right", 1000) if depth_right else None
    return root


def complete_tree(n):
    return build_tree(list(range(n)))


rng = random.Random(500)
trees = [random_tree(rng, 500, 0.0), random_tree(rng, 500, 0.95), chain_tree(500, "left"), chain_tree(500, "right"),
         complete_tree(500), v_tree(200, 200), v_tree(250, 100)]
for root in trees:
    fast = [IMPLS[name](root) for name in ("pair", "walk", "climb", "Solution", "SolutionPair", "SolutionWalk", "SolutionClimb")]
    assert all(x is fast[0] for x in fast)
    if len(all_nodes(root)) <= 220:
        assert deepest_brute(root) is fast[0]
    assert fast[0] is oracle(root) if len(all_nodes(root)) <= 260 else True
print("7 cây lớn: 7 cài đặt cùng trả một nút; cây V cân (200,200) trả gốc:", Solution().subtreeWithAllDeepest(trees[5]) is trees[5],
      "| V lệch (250,100) trả lá cuối nhánh trái:", Solution().subtreeWithAllDeepest(trees[6]).val)

7 cây lớn: 7 cài đặt cùng trả một nút; cây V cân (200,200) trả gốc: True | V lệch (250,100) trả lá cuối nhánh trái: 250


## 13. Các lỗi người mới hay gặp

### Lỗi 1: khi hoà vẫn đi sang trái

Dùng `hl >= hr` rồi trả đáp án của bên trái, bỏ qua rằng hoà nghĩa nút sâu nhất có ở **cả hai** bên.

In [22]:
r = build_tree([1, 2, 3])
print("hoà đi trái:", bug_tie_left(r).val, "| đúng:", Solution().subtreeWithAllDeepest(r).val)
assert bug_tie_left(r) is not r and Solution().subtreeWithAllDeepest(r) is r

hoà đi trái: 2 | đúng: 1


**Vì sao sai:** nút sâu nhất `2` và `3` ở hai phía, cây con nhỏ nhất chứa cả hai là cây con của gốc `1`. **Đúng:** `hl == hr` → trả **nút hiện tại**.

### Lỗi 2: trả về lá sâu nhất đầu tiên

Chỉ đúng khi nút sâu nhất **duy nhất**.

In [23]:
r = build_tree(examples[0][0])
print("lá sâu đầu tiên:", bug_first_deepest_leaf(r).val, "| đúng:", Solution().subtreeWithAllDeepest(r).val)
assert bug_first_deepest_leaf(r).val == 7 and Solution().subtreeWithAllDeepest(r).val == 2

lá sâu đầu tiên: 7 | đúng: 2


**Vì sao sai:** có hai nút sâu nhất `7`, `4`: cây con nhỏ nhất chứa **cả hai** là của `2`. **Đúng:** LCA của tập nút sâu nhất.

### Lỗi 3: LCA của mọi lá

Các lá nông không phải "sâu nhất"; chúng làm LCA bị đẩy lên cao.

In [24]:
r = build_tree([1, 2, 3, 4, None, None, None, 5])
print("LCA mọi lá:", bug_lca_all_leaves(r).val, "| đúng:", Solution().subtreeWithAllDeepest(r).val)
assert bug_lca_all_leaves(r).val == 1 and Solution().subtreeWithAllDeepest(r).val == 5

LCA mọi lá: 1 | đúng: 5


**Vì sao sai:** lá `3` nông nên không thuộc tập sâu nhất. **Đúng:** chỉ xét các lá ở **độ sâu lớn nhất**.

### Lỗi 4: đi theo nhánh sâu hơn tới tận lá

Quên dừng khi hoà.

In [25]:
r = build_tree(examples[0][0])
print("đi tới lá:", bug_follow_taller_to_leaf(r).val, "| đúng:", Solution().subtreeWithAllDeepest(r).val)
assert bug_follow_taller_to_leaf(r).val != 2

đi tới lá: 7 | đúng: 2


**Vì sao sai:** tại `2` hai bên sâu bằng nhau nên phải **dừng** ở `2`. **Đúng:** dừng khi `hl == hr`.

### Lỗi 5: so sánh theo giá trị thay vì danh tính

Trong test, nếu cây có giá trị trùng (không hợp lệ trong đề, nhưng xảy ra khi tự sinh test), so `got.val == want.val` có thể đúng "may". Hãy so bằng `is`. Minh hoạ: hai nút khác nhau cùng giá trị.

In [26]:
a, b = TreeNode(7), TreeNode(7)
print("a.val == b.val:", a.val == b.val, "| a is b:", a is b)
assert a.val == b.val and a is not b

a.val == b.val: True | a is b: False


**Vì sao quan trọng:** đề yêu cầu trả **nút cụ thể** trong cây; hai nút có thể giống giá trị nhưng khác vị trí. **Đúng:** so `is` trong test; trên LeetCode hệ thống so sánh theo cấu trúc/giá trị vì giá trị duy nhất.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "nút/cây con nhỏ nhất chứa mọi nút sâu nhất" = **LCA của các lá sâu nhất**. Mẫu "DFS trả `(chiều cao, đáp án)`" giải gọn các bài về độ sâu/chiều cao kèm "nút đại diện".

**Mẹo:**

1. Phân biệt **độ sâu** (từ gốc xuống) và **chiều cao** (từ nút xuống lá).
2. So chiều cao hai con: hơn → đi bên đó; hoà → dừng.
3. Trả về **cặp** để gộp hai lượt thành một.
4. BFS tầng cuối + bản đồ cha = cách "leo lên tới khi gặp nhau".
5. Test bằng `is`, oracle bằng định nghĩa (cây con nhỏ nhất chứa tập sâu nhất).

**Họ bài cùng pattern:** Lowest Common Ancestor of Deepest Leaves (1123, y hệt), Lowest Common Ancestor of a Binary Tree (236), Maximum Depth of Binary Tree (104), Diameter of Binary Tree (543), Binary Tree Maximum Path Sum (124), Balanced Binary Tree (110).

## 15. Biến thể

1. **LCA của tập nút bất kỳ** (236 tổng quát).
2. **Chiều cao cây con kết quả** (1123 hỏi giá trị / cây con).
3. **Đếm số nút sâu nhất.**
4. **Smallest subtree chứa mọi nút có giá trị lớn nhất** (tập tuỳ ý).
5. **Độ sâu của đáp án** (`depth(ans)`).

Cài biến thể 1 (`lca_of_set`) và kiểm rằng với tập nút sâu nhất trả đúng đáp án bài 865; đối chiếu 3 và 5:

In [27]:
def lca_of_set(root, targets):
    """LCA của một tập nút (theo danh tính). Trả (số nút target trong cây con, nút) bằng DFS."""
    targets = set(targets)
    answer = [None]

    def dfs(node):
        if node is None:
            return 0
        cnt = dfs(node.left) + dfs(node.right) + (1 if node in targets else 0)
        if cnt == len(targets) and answer[0] is None:     # nút ĐẦU TIÊN (từ dưới lên) chứa đủ tập là LCA
            answer[0] = node
        return cnt

    dfs(root)
    return answer[0]


def count_deepest(root):
    dep = depths(root)
    mx = max(dep.values())
    return sum(1 for d in dep.values() if d == mx)


rng = random.Random(15)
for _ in range(400):
    root = random_tree(rng, rng.randint(1, 30), rng.choice([0.0, 0.5, 0.9]))
    dep = depths(root)
    mx = max(dep.values())
    deepest = [n for n, d in dep.items() if d == mx]
    ans = Solution().subtreeWithAllDeepest(root)
    assert lca_of_set(root, deepest) is ans
    assert count_deepest(root) == len(deepest)
    assert dep[ans] + height(ans) == mx + 1                    # độ sâu đáp án + chiều cao cây con = tổng số tầng
print("lca_of_set(tập sâu nhất) = đáp án bài 865; dep(ans) + height(ans) = số tầng; đếm nút sâu nhất đúng (400 cây)")

lca_of_set(tập sâu nhất) = đáp án bài 865; dep(ans) + height(ans) = số tầng; đếm nút sâu nhất đúng (400 cây)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Theo định nghĩa | kiểm từng nút có chứa hết tập sâu nhất | `O(n²)` | `O(n)` | oracle phụ |
| 2. DFS `(h, nút)` | hơn → bên đó; hoà → nút này | `O(n)` | `O(h)` | **đáp án chính** |
| 3. Chiều cao + đi xuống | `dict` chiều cao, đi theo nhánh sâu hơn | `O(n)` | `O(n)` | dễ giải thích |
| 4. BFS + leo cha | tầng cuối, leo lên tới khi còn một nút | `O(n)` | `O(n)` | không đệ quy |

Cả bốn trả đúng nút oracle trên 2 055 hình dạng cây nhỏ, 1 500 cây ngẫu nhiên và các cây 500 nút (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [28]:
for vals in ([1, 2, 3], [1, 2, None, 3], [1, 2, 3, 4, 5, None, 6, 7]):
    r = build_tree(vals)
    print(vals, "->", to_level(Solution().subtreeWithAllDeepest(r)))

[1, 2, 3] -> [1, 2, 3]
[1, 2, None, 3] -> [3]
[1, 2, 3, 4, 5, None, 6, 7] -> [7]


### Bài 2 (tính tay)

Cây `[1,2,3,4,5,6,7,8,null,null,null,null,null,null,9]`: độ sâu mỗi nút? nút sâu nhất? cây con nhỏ nhất chứa chúng? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DFS trả cặp:

In [29]:
def pair_todo(root):
    def dfs(node):
        if node is None:
            return 0, None
        hl, nl = dfs(node.left)
        hr, nr = dfs(node.right)
        # TODO: so hl và hr; trả (chiều cao mới, nút ứng viên)
        return 0, None

    return dfs(root)[1]


# Khi làm xong, bỏ comment để kiểm tra:
# assert to_level(pair_todo(build_tree(examples[0][0]))) == [2, 7, 4]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **BFS tầng cuối + leo cha**:

In [30]:
class MySolution:
    def subtreeWithAllDeepest(self, root):
        pass            # TODO


# Khi làm xong:
# assert to_level(MySolution().subtreeWithAllDeepest(build_tree(examples[0][0]))) == [2, 7, 4]

### Bài 5 (mini challenge): chiều cao và độ sâu của đáp án

Viết `deepest_info(root)` trả `(độ sâu của nút đáp án, chiều cao cây con của đáp án, số nút sâu nhất)` trong **một** lượt DFS (không gọi lại `height`).

In [31]:
def deepest_info(root):
    pass            # TODO


# Khi làm xong:
# r = build_tree(examples[0][0]); assert deepest_info(r) == (2, 2, 2)

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: so chiều cao trái và phải ở mỗi nút.
- Mức 2: `[1,2,3]` hoà → `1`; `[1,2,None,3]` chỉ trái → `3`.
- Mức 3: kết quả `[1, 2, 3]`, `[3]`, `[7]`: ở cây thứ ba, `7` là con của `4` và là nút sâu nhất duy nhất (độ sâu 3) nên đáp án là chính `7`.

**Bài 2**

- Mức 1: các nút `8` (con trái của `4`) và `9` (con phải của `7`) cùng độ sâu 3.
- Mức 2: `8` thuộc nhánh `2 → 4 → 8`; `9` thuộc nhánh `3 → 7 → 9`: hai nhánh khác nhau của gốc.
- Mức 3: đáp án là gốc `1`.

**Bài 3**

- Mức 1: ba trường hợp `hl > hr`, `hl < hr`, `hl == hr`.
- Mức 2: trả `(hl + 1, nl)`, `(hr + 1, nr)`, `(hl + 1, node)`.
- Mức 3: xem Cách 2.

**Bài 4**

- Mức 1: BFS theo tầng, ghi `parent`.
- Mức 2: tầng cuối thành tập; leo lên bằng tập cha.
- Mức 3: dừng khi tập chỉ còn một phần tử.

**Bài 5**

- Mức 1: DFS trả `(chiều cao, nút, số nút sâu nhất ở tầng cuối của cây con)`.
- Mức 2: số nút sâu nhất: `hl > hr` → của trái; `<` → của phải; `==` → tổng hai bên (hoặc 1 nếu cả hai rỗng).
- Mức 3: độ sâu đáp án: chạy lại tìm nút (truyền độ sâu khi duyệt) hoặc lưu `depth` trong lúc đệ quy.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [32]:
expect = {(1, 2, 3): [1, 2, 3], (1, 2, None, 3): [3], (1, 2, 3, 4, 5, None, 6, 7): [7]}
for vals, want in expect.items():
    assert to_level(Solution().subtreeWithAllDeepest(build_tree(list(vals)))) == want == to_level(oracle(build_tree(list(vals))))
print("Bài 1 OK")

Bài 1 OK


### Bài 2

Các nút `8` và `9` cùng độ sâu `3`, ở hai nhánh khác nhau của gốc → đáp án là gốc `1` (cả cây).

In [33]:
vals = [1, 2, 3, 4, 5, 6, 7, 8, None, None, None, None, None, None, 9]
r = build_tree(vals)
dep = depths(r)
assert sorted(n.val for n, d in dep.items() if d == 3) == [8, 9]
assert Solution().subtreeWithAllDeepest(r) is r is oracle(r)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [34]:
def pair_answer(root):
    def dfs(node):
        if node is None:
            return 0, None
        hl, nl = dfs(node.left)
        hr, nr = dfs(node.right)
        if hl > hr:
            return hl + 1, nl
        if hl < hr:
            return hr + 1, nr
        return hl + 1, node

    return dfs(root)[1]


assert to_level(pair_answer(build_tree(examples[0][0]))) == [2, 7, 4]
rng = random.Random(3)
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 25), 0.5)
    assert pair_answer(root) is oracle(root)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [35]:
class MySolutionAnswer:
    def subtreeWithAllDeepest(self, root):
        parent = {root: None}
        level = [root]
        while True:
            nxt = []
            for node in level:
                for ch in (node.left, node.right):
                    if ch:
                        parent[ch] = node
                        nxt.append(ch)
            if not nxt:
                break
            level = nxt
        cur = set(level)
        while len(cur) > 1:
            cur = {parent[x] for x in cur}
        return cur.pop()


assert to_level(MySolutionAnswer().subtreeWithAllDeepest(build_tree(examples[0][0]))) == [2, 7, 4]
rng = random.Random(4)
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 25), 0.5)
    assert MySolutionAnswer().subtreeWithAllDeepest(root) is oracle(root)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

DFS trả `(chiều cao, nút đáp án, số nút sâu nhất)`. Độ sâu của nút đáp án suy ra từ chiều cao: các nút sâu nhất nằm ở tầng cuối, cây con của đáp án có `h_ans` tầng nên đáp án nằm ở độ sâu `tổng số tầng − h_ans`.

In [36]:
def deepest_info_answer(root):
    def dfs(node):
        if node is None:
            return 0, None, 0
        hl, nl, cl = dfs(node.left)
        hr, nr, cr = dfs(node.right)
        if hl > hr:
            return hl + 1, nl, cl
        if hl < hr:
            return hr + 1, nr, cr
        return hl + 1, node, (cl + cr) or 1

    total, ans, cnt = dfs(root)
    h_ans = height(ans)
    return total - h_ans, h_ans, cnt                       # độ sâu = tổng số tầng - chiều cao cây con của đáp án


r = build_tree(examples[0][0])
assert deepest_info_answer(r) == (2, 2, 2)
rng = random.Random(5)
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 30), 0.5)
    d, h, c = deepest_info_answer(root)
    ans = oracle(root)
    assert (d, h, c) == (depths(root)[ans], height(ans), count_deepest(root))
print("Bài 5 OK: (độ sâu, chiều cao, số nút sâu nhất) khớp tính trực tiếp")

Bài 5 OK: (độ sâu, chiều cao, số nút sâu nhất) khớp tính trực tiếp


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template ("nút đại diện" theo chiều cao, DFS trả cặp):**

```python
class Solution:
    def subtreeWithAllDeepest(self, root):
        def dfs(node):
            if node is None:
                return 0, None
            hl, nl = dfs(node.left)
            hr, nr = dfs(node.right)
            if hl > hr:
                return hl + 1, nl
            if hl < hr:
                return hr + 1, nr
            return hl + 1, node          # hoà: nút này là LCA của các nút sâu nhất
        return dfs(root)[1]
```

**Checklist trước khi submit:**

- [ ] Hoà (`hl == hr`) trả **nút hiện tại**, không đi tiếp xuống một bên?
- [ ] `None` có chiều cao `0` và đáp án `None`; lá hoà (`0 == 0`) trả chính lá?
- [ ] Phân biệt chiều cao (từ nút xuống) với độ sâu (từ gốc xuống)?
- [ ] Chỉ xét các nút **sâu nhất** (không phải mọi lá)?
- [ ] Đã thử cây một nút, chuỗi trái/phải, cây đầy đủ, hai lá sâu nhất khác phía, ví dụ 1 → nút `2`?
- [ ] Đúng tên `class Solution`, `subtreeWithAllDeepest(self, root)`, không in debug.